# Benders Decomposition — Before, Principle, Application, and Effect

For models that naturally divide into a "design/location decision part" (whether to open facilities) and an "operational decision part based on those given parameters" (transportation), instead of packing both into a single MIP and solving it altogether (monolithic), they can be iteratively solved by separating them into a **master problem (a few linking variables)** and a **subproblem (the rest, usually an LP)**.
By teaching the master problem only the "sensitivity" obtained from the subproblem's duals as cuts, the lower bound and upper bound converge while alternating back and forth — this is Benders Decomposition.

This notebook tracks this phenomenon and its remedy through the following pattern:

1. **Issue (before)** — Solve facility location directly as a single monolithic MIP (checking the structure of linking constraints).
2. **Principle (principle)** — See the cut alternation between master and subproblem in the convergence curve of actual measured data.
3. **Application (how)** — Check `mk.benders(master_build, subproblem_solve)` with a minimal configuration.
4. **Effect (before/after)** — Compare the monolithic direct solve vs the final Benders master problem (with all cuts added).

The subject is **Capacitated Facility Location** (the same mathematical structure as `samples/location_and_network_design/facility.py`: opening $y_i \in \{0,1\}$ / transportation $x_{ij} \ge 0$). However, because the bundled sample is too small with 4 facilities and 5 customers (it does not even meet the `decomposable` diagnosis firing condition `max_linking_groups>=4 and n_heavy_linking<=3`, and there are combinations of `y` for which the subproblem is infeasible at this scale), we will generate a **synthetic instance with the same mathematical structure where scale can be controlled** within this notebook for verification (CLAUDE.md policy: verify by creating a verifiable problem yourself).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/location_and_network_design"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum
from scipy.optimize import linprog

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import facility as fac

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Issue (before) — Solving facility location monolithically

With the same structure as `facility.py` (fixed costs `fixed_i` / capacities `cap_i` / transport costs `cost_ij` / demands `demand_j`), we synthesize an instance with 14 facilities, 20 customers, and a maximum opening limit of 6 (reproducible with a fixed random seed). Capacities are generated with a safety margin so that **"the total capacity of any combination of 6 open facilities exceeds total demand"** (otherwise, Benders could pick an infeasible `y` right away without cuts. This corresponds to the simplification that `mk.benders` does not handle feasibility cuts, which will be touched on again).

In [ ]:
def gen_facility(n_fac=14, n_cust=20, open_max=6, seed=0):
    rng = np.random.default_rng(seed)
    demand = rng.integers(10, 40, n_cust)
    total_demand = int(demand.sum())
    cap_center = total_demand / open_max * 1.35
    cap = rng.integers(int(cap_center * 0.85), int(cap_center * 1.15), n_fac)
    fixed = rng.integers(80, 220, n_fac)
    cost = rng.integers(1, 15, size=(n_fac, n_cust))
    facs = [f"F{i}" for i in range(n_fac)]
    custs = [f"C{j}" for j in range(n_cust)]
    return dict(facs=facs, custs=custs, fixed=dict(zip(facs, fixed)),
                cap=dict(zip(facs, cap)), demand=dict(zip(custs, demand)),
                cost={(facs[i], custs[j]): int(cost[i, j])
                      for i in range(n_fac) for j in range(n_cust)},
                open_max=open_max)

def build_monolithic(data):
    m = Model("facility_mono")
    facs, custs = data["facs"], data["custs"]
    y = {i: m.addVar(vtype="B", name=f"y_{i}") for i in facs}
    x = {(i, j): m.addVar(lb=0, name=f"x_{i}_{j}") for i in facs for j in custs}
    for j in custs:
        m.addCons(quicksum(x[i, j] for i in facs) >= data["demand"][j], name=f"demand_{j}")
    for i in facs:
        m.addCons(quicksum(x[i, j] for j in custs) <= data["cap"][i] * y[i], name=f"capacity_{i}")
    m.addCons(quicksum(y[i] for i in facs) <= data["open_max"], name="open_limit")
    m.setObjective(quicksum(data["fixed"][i] * y[i] for i in facs)
                    + quicksum(data["cost"][i, j] * x[i, j] for i in facs for j in custs), "minimize")
    m.data = dict(x=x, y=y)
    return m

data = gen_facility()
mono = build_monolithic(data)
mono.hideOutput()
mono.setParam("limits/time", 30)
mono.optimize()
mono_obj, mono_nodes, mono_time = mono.getObjVal(), mono.getNNodes(), mono.getSolvingTime()
print(f"monolithic: status={mono.getStatus()} obj={mono_obj:.1f} "
      f"nodes={mono_nodes} time={mono_time:.3f}s")

Even at this scale, SCIP finds the optimal solution at the root node alone (it's small, so the monolithic solve is not slow itself — as pointed out in FINDINGS: "for small scale, solving a single problem directly can be faster").
Nevertheless, the structure is clearly two blocks (opening $y$ / independent transportation $x_{i\cdot}$ for each facility), and the block structure metrics from `mk.analyze` show the state of the linkages.

In [ ]:
report = mk.analyze(lambda: build_monolithic(data), name="facility-synth", time_limit=15)
print(report.summary())
print("max_linking_groups:", report.metrics.get("max_linking_groups"),
      " n_heavy_linking:", report.metrics.get("n_heavy_linking"))

## 2. Principle (principle) — Measured Convergence Curve of Cut Loops

The master problem holds the opening variables $y$ (integer) and a lower bound for transportation costs $\eta$: $\min \sum_i \mathrm{fixed}_i\, y_i + \eta$.
The subproblem is a transportation LP with a fixed $\hat y$. From the duals $g_i$ of the subproblem's capacity constraints, an **optimality cut** $\eta \ge Q(\hat y) + \sum_i g_i (y_i - \hat y_i)$ is created and added to the master problem.

```
  Master problem min c·y+η ──fix ŷ──▶ Subproblem min transport cost Q(ŷ)
       ▲                              │
       └──── η ≥ Q(ŷ)+Σg·(y-ŷ) ───────┘
  Repeat until Lower Bound (master obj) and Upper Bound (actual Q) match
```

The transportation LP only has **aggregate capacity constraints** per facility (no limits per arc), so we can use the classical solvability theorem for transportation problems: if "total capacity ≥ total demand", it is always feasible.
By adding this directly to the master problem in advance ($\sum_i \mathrm{cap}_i\, y_i \ge \text{Total Demand}$), we can safely iterate without feasibility cuts, which `mk.benders` does not handle.

In [ ]:
def make_master_build(data):
    facs = data["facs"]
    total_demand = sum(data["demand"].values())

    def master_build(cuts):
        m = Model("facility_master")
        y = {i: m.addVar(vtype="B", name=f"y_{i}") for i in facs}
        eta = m.addVar(lb=0, name="eta")
        m.addCons(quicksum(y[i] for i in facs) <= data["open_max"], name="open_limit")
        m.addCons(quicksum(data["cap"][i] * y[i] for i in facs) >= total_demand, name="agg_capacity")
        for k, cut in enumerate(cuts):
            m.addCons(eta >= cut["Q"] + quicksum(cut["grad"][i] * (y[i] - cut["yhat"][i])
                                                 for i in facs), name=f"cut_{k}")
        m.setObjective(quicksum(data["fixed"][i] * y[i] for i in facs) + eta, "minimize")
        m.data = dict(eta=eta, y=y)
        return m
    return master_build

def make_subproblem_solve(data):
    facs, custs = data["facs"], data["custs"]
    n, p = len(facs), len(custs)

    def subproblem_solve(y_hat):
        c = np.array([data["cost"][i, j] for i in facs for j in custs], dtype=float)
        A_ub = np.zeros((p + n, n * p))
        b_ub = np.zeros(p + n)
        idx = lambda a, b: a * p + b
        for jb, j in enumerate(custs):
            for ia in range(n):
                A_ub[jb, idx(ia, jb)] = -1.0
            b_ub[jb] = -data["demand"][j]
        for ia, i in enumerate(facs):
            for jb in range(p):
                A_ub[p + ia, idx(ia, jb)] = 1.0
            b_ub[p + ia] = data["cap"][i] * y_hat[i]
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method="highs")
        Q = float(res.fun)
        duals_cap = res.ineqlin.marginals[p:p + n]
        grad = {i: float(data["cap"][i] * duals_cap[ia]) for ia, i in enumerate(facs)}
        return Q, grad

    return subproblem_solve

_captured_cuts = []
def _master_build_capturing(cuts):
    _captured_cuts.clear()
    _captured_cuts.extend(cuts)
    return make_master_build(data)(cuts)

subproblem_solve = make_subproblem_solve(data)
res = mk.benders(_master_build_capturing, subproblem_solve, max_iter=200, tol=1e-6)
print(f"benders: lb={res['lb']:.1f} ub={res['ub']:.1f} cuts={res['n_cuts']} "
      f"iters={len(res['history'])}  monolithicと一致={abs(res['ub']-mono_obj) < 1e-4}")

In [ ]:
its = [h["iter"] for h in res["history"]]
fig = go.Figure(layout=base_layout(
    "ベンダーズ反復の収束 — 下界(主問題)が最適性カットで上界に到達", "反復(=追加した最適性カット数)",
    "目的関数値(総費用)", height=420))
fig.add_trace(go.Scatter(x=its, y=[h["ub"] for h in res["history"]], mode="lines+markers",
    name="上界 UB(最良の実行可能解)", line=dict(color=C["warn"], width=2),
    marker=dict(size=6, color=C["warn"]),
    hovertemplate="反復%{x}<br>UB %{y:.1f}<extra></extra>"))
fig.add_trace(go.Scatter(x=its, y=[h["lb"] for h in res["history"]], mode="lines+markers",
    name="下界 LB(主問題)", line=dict(color=C["s1"], width=2),
    marker=dict(size=6, color=C["s1"]),
    hovertemplate="反復%{x}<br>LB %{y:.1f}<extra></extra>"))
fig.add_hline(y=mono_obj, line=dict(color=C["muted"], width=2, dash="dash"),
              annotation_text=f"monolithicの最適値 {mono_obj:.0f}", annotation_position="right",
              annotation_font=dict(color=C["ink2"], size=11))
show(fig)

The LB (master problem, which tightens as cuts increase) and UB (the best subproblem cost at that point) converge to the monolithic optimal value by sandwiching it (for actual values, see the output of the next cell). Only one transportation LP is solved per cut, and the master problem remains a small MIP with only 14 opening variables + $\eta$ at all times.

In [ ]:
print(f"収束: {len(res['history'])}反復・{res['n_cuts']}カットで "
      f"LB={res['lb']:.1f} = UB={res['ub']:.1f} = monolithic最適値{mono_obj:.1f}")

## 3. Application (how) — `mk.benders(master_build, subproblem_solve)`

The problem-specific parts are just two callbacks: `master_build(cuts) -> Model` (places `eta`/`y` into `model.data`) and `subproblem_solve(y_hat) -> (Q, grad)`. Here we check the API contract with a minimal toy example (2 facilities, 2 customers) — verifying only that `mk.benders` matches the monolithic direct solve in a trivial setting where capacity is always sufficient.

In [ ]:
toy = gen_facility(n_fac=2, n_cust=2, open_max=1, seed=1)
toy_mono = build_monolithic(toy)
toy_mono.hideOutput(); toy_mono.optimize()

toy_res = mk.benders(make_master_build(toy), make_subproblem_solve(toy), max_iter=50, tol=1e-6)
print(f"toy monolithic obj={toy_mono.getObjVal():.2f}  "
      f"toy benders ub={toy_res['ub']:.2f}  一致={abs(toy_mono.getObjVal()-toy_res['ub']) < 1e-4}  "
      f"(反復{len(toy_res['history'])}・カット{toy_res['n_cuts']})")

## 4. Effect (before/after) — Monolithic vs "Final Master Problem with All Cuts"

We pass to `mk.compare_variants` (a) a build_fn that solves the monolithic model directly, and (b) a build_fn for the master problem containing the final set of cuts collected in Section 3 (which lacks the 280 transportation variables and only has the 14 opening variables + $\eta$), and compare their root dual bounds, final gaps, and node counts. While "making the master problem smaller" itself intuitively helps, whether it can reach optimality **from the root** using only cuts is a separate issue (cuts are just a collection of supporting hyperplanes created at specific integer points, and it's not guaranteed how tight their convex hull is in the LP relaxation). Here, we measure both honestly.

In [ ]:
final_cuts = list(_captured_cuts)

def build_master_with_final_cuts():
    return make_master_build(data)(final_cuts)

df = mk.compare_variants(
    {"monolithic": lambda: build_monolithic(data),
     "benders最終主問題(全カット搭載)": build_master_with_final_cuts},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
mono_row = df.loc[df["variant"] == "monolithic"].iloc[0]
bend_row = df.loc[df["variant"] == "benders最終主問題(全カット搭載)"].iloc[0]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["monolithic", "benders最終主問題"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [mono_row["root_dual"], bend_row["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [mono_row["final_gap"]*100, bend_row["final_gap"]*100], lambda v: f"{v:.1f}%")
add_bars(3, [mono_row["nodes"], bend_row["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="ベンダーズ分解の before / after(monolithic vs 全カット搭載の最終主問題)",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [ ]:
print(f"ルート双対境界 : monolithic {mono_row['root_dual']:.1f}  vs  "
      f"benders最終主問題 {bend_row['root_dual']:.1f}")
print(f"最終gap        : monolithic {mono_row['final_gap']:.1%}  vs  "
      f"benders最終主問題 {bend_row['final_gap']:.1%}")
print(f"ノード数       : monolithic {int(mono_row['nodes'])}  vs  "
      f"benders最終主問題 {int(bend_row['nodes'])}")


## Summary

- **Even after completely removing transportation variables (14 facilities × 20 customers = 280 continuous variables) from the master problem, it strictly converges to the same optimal value of 1537 as the monolithic model through cut iterations alone** (convergence curve in Section 2).
  This is the core effect of Benders, and it holds true for this instance as well.
- On the other hand, "the node count when solving the cut-added master problem alone" does not necessarily fall below monolithic (Section 4). Cuts are merely a collection of supporting hyperplanes created at specific integer `ŷ`, and how tight their convex hull is at the LP relaxation (root) depends on the structure. For this instance, monolithic itself was so "easy" that it could be solved in 1 node at the root, making the branching costs of the cut master problem appear relatively high.

### Why SCIP Doesn't Do It Automatically

SCIP solves the given MIP exactly as is (using Big-M, cuts, and branching). The **decomposition approach itself** — "divide the problem into linking variables and subproblems, and feed the subproblem's duals back to the master problem as cuts" — is an architectural decision made by the modeler, and cannot be automatically extracted from a single built MIP.
This is why the diagnosis points out block structures (few linking constraints) and the modeler writes `master_build`/`subproblem_solve` in a division of labor.

### When It Is Ineffective / Cautions

- The implemented `mk.benders` **does not handle feasibility cuts**. In this notebook, we bypassed this by adding "aggregate capacity ≥ total demand" directly to the master problem (a simplification possible because of the special structure where the transportation LP lacks per-arc upper bounds). For more general structures (e.g., with arc capacities), a `ŷ` for which the subproblem is infeasible could occur, requiring extensions.
- For small scales, directly solving the monolithic model can sometimes be faster (even in this notebook's instance, monolithic is solved at the root node). The value of decomposition lies in its **scalability** — keeping the master problem small and solving subproblems independently (in parallel).

Related: [Methods Guide 5. Benders Decomposition](../../playbook/05-benders.md) /
API [`mk.benders`](../../api/frameworks.md) / worked example: `experiments/run_benders.py`